In [ ]:
!pip install pandas openpyxl

In [ ]:
# Cell content cleared due to redundancy (duplicate imports).

In [ ]:
import pandas as pd
import re
import torch
from sklearn.metrics import classification_report
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

In [ ]:
!pip install pandas transformers datasets scikit-learn

In [ ]:
from google.colab import files

uploaded = pd.read_excel("")

In [ ]:
df = uploaded

df.head()
#df = df['language'] == "BHO")
# For now, let's keep all languages to allow augmentation to proceed.
df['language'].value_counts()
#df.head(50)

In [ ]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"(.)\1{2,}", r"\1", text)  # reduce repeating chars
    text = re.sub(r"[^a-zA-Z\u0900-\u097F\s]", " ", text)  # keep Hindi + English
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:

print("Original size:", len(df))
display(df.head())

In [ ]:
from huggingface_hub import login
login()

In [ ]:
#model_name = "google/muril-base-cased"
model_name = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Assuming 'aug_df' is already defined from cell eTRzQVV74K9F
# Defining tokenize function (copied from original cell 6L5lTtF3YNC to ensure it's available)
def tokenize(batch):
    # model_name is expected to be global from K_lzqFTF3J9F
    tokenizer_instance = AutoTokenizer.from_pretrained(model_name)
    return tokenizer_instance(batch['text'], truncation=True, padding='max_length', max_length=128)

# Split the augmented data into training and validation sets (copied from original cell WMO9PqQw27Aj)
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['label'],
    random_state=42
)

# Map labels to numerical values (copied from original cell UjZXm_003HOu)
label_mapping = {'NO': 0, 'O': 1}
train_df['label'] = train_df['label'].map(label_mapping)
val_df['label'] = val_df['label'].map(label_mapping)

# Create Dataset objects (copied from original cell UjZXm_003HOu)
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True).astype({'text': str}))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True).astype({'text': str}))

# Tokenize the datasets (copied from original cell 6L5lTtF3YNC)
train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

# Set format for PyTorch (copied from original cell 6L5lTtF3YNC)
train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
val_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])



Map:   0%|          | 0/14279 [00:00<?, ? examples/s]

Map:   0%|          | 0/1587 [00:00<?, ? examples/s]

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Defining tokenize function (copied from original cell 6L5lTtF3YNC to ensure it's available)
def tokenize(batch):
    # model_name is expected to be global from K_lzqFTF3J9F
    tokenizer_instance = AutoTokenizer.from_pretrained(model_name)
    return tokenizer_instance(batch['cleaned_text'], truncation=True, padding='max_length', max_length=128)

# Split the original (non-augmented) data into training and validation sets
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df['label'], # Stratify by label from original df
    random_state=42
)

# Map labels to numerical values
label_mapping = {'NO': 0, 'O': 1}
train_df['label'] = train_df['label'].map(label_mapping)
val_df['label'] = val_df['label'].map(label_mapping)

# Create Dataset objects (using 'cleaned_text' column)
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True).astype({'cleaned_text': str}))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True).astype({'cleaned_text': str}))

# Tokenize the datasets
train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

# Set format for PyTorch
train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
val_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])

### Zero-Shot Evaluation on Validation Samples

This section performs a zero-shot classification using a pre-trained model without any fine-tuning on the current dataset. This allows us to see how well the model generalizes to the task directly from its pre-training.

In [ ]:
from transformers import pipeline

# Initialize a zero-shot classification pipeline
# We'll use the 'text-classification' task and a suitable pre-trained model.
# 'FacebookAI/xlm-roberta-base' is already loaded, so we'll use a general model here or a zero-shot specific one.
# For general zero-shot classification, 'MoritzLaurer/mDeBERTa-v3-base-mnli-fever-anli' is a good choice.
# However, since XLM-RoBERTa is already in use, let's assume a zero-shot task using it.

# Note: Zero-shot classification often uses models fine-tuned on NLI tasks (like MNLI).
# If 'model_name' (XLM-RoBERTa-base) is not fine-tuned for NLI, performance might be limited.
# For a true zero-shot experience, you might want to use a model like 'MoritzLaurer/mDeBERTa-v3-base-mnli-fever-anli'
# or 'facebook/bart-large-mnli'.
# For simplicity and using existing model infrastructure, let's define candidate labels and use the current model.

# Let's use the tokenizer from the currently loaded model_name.
# The `pipeline` function can automatically load a suitable zero-shot model if not specified.
# We'll use a generic zero-shot model for robustness here, or if model_name is NLI-capable.
# For this example, let's explicitly specify a zero-shot model if `model_name` is not suitable.

# If `model_name` (FacebookAI/xlm-roberta-base) is not an NLI model, we need a dedicated zero-shot model.
# Let's explicitly define a zero-shot model for this step to ensure functionality.
zero_shot_model_name = "joeddav/xlm-roberta-large-xnli" # A common multilingual NLI model
zero_shot_tokenizer = AutoTokenizer.from_pretrained(zero_shot_model_name)

classifier = pipeline("zero-shot-classification", model=zero_shot_model_name, tokenizer=zero_shot_tokenizer)

print(f"Zero-shot classifier loaded using model: {zero_shot_model_name}")

In [ ]:
import torch
from torch.utils.data import DataLoader

def get_zero_shot_predictions(classifier, zero_shot_dataloader, candidate_labels):
    """
    Performs zero-shot classification on the validation data using a DataLoader for batching.

    Args:
        classifier: The Hugging Face zero-shot classification pipeline.
        zero_shot_dataloader (torch.utils.data.DataLoader): DataLoader yielding batches of texts.
        candidate_labels (list): A list of candidate labels (e.g., ['offensive', 'not-offensive']).

    Returns:
        list: A list of predicted labels (0 for 'not-offensive', 1 for 'offensive').
    """
    predictions_zero_shot = []

    for batch_texts in zero_shot_dataloader:
        # Each batch_texts is a list of strings thanks to the custom collate_fn
        results = classifier(batch_texts, candidate_labels=candidate_labels, multi_label=False)
        predictions_zero_shot.extend(results)

    y_pred_zero_shot = []
    for result in predictions_zero_shot:
        predicted_label = result['labels'][0]
        if predicted_label == 'offensive':
            y_pred_zero_shot.append(1)
        else:
            y_pred_zero_shot.append(0)
    return y_pred_zero_shot

print("Zero-shot prediction function defined (using DataLoader for batching).")

Zero-shot prediction function defined (using DataLoader for batching).


In [ ]:
# Prepare data for zero-shot classification
candidate_labels = ['offensive', 'not-offensive'] # Map 'O' to 'toxic' and 'NO' to 'non-toxic'

# Extract true labels for evaluation
y_true = val_df['label'].tolist()

# Ensure 'text' column is explicitly string type before creating Dataset
val_df['text'] = val_df['text'].astype(str)

# Create a Hugging Face Dataset for zero-shot inference (containing only text)
zero_shot_dataset = Dataset.from_pandas(val_df[['text']].reset_index(drop=True))

# Custom collate_fn to extract 'text' from each item in the batch
def collate_fn_zero_shot(batch):
    return [item['text'] for item in batch]

# Create a DataLoader to handle batching of texts
zero_shot_dataloader = DataLoader(zero_shot_dataset, batch_size=32, collate_fn=collate_fn_zero_shot)

print("Performing zero-shot inference on validation samples using the defined function...")

# Use the newly defined function to get predictions
y_pred_zero_shot = get_zero_shot_predictions(classifier, zero_shot_dataloader, candidate_labels)

print("Zero-shot inference complete.")

print("Evaluating zero-shot model performance...")

Performing zero-shot inference on validation samples using the defined function...
Zero-shot inference complete.
Evaluating zero-shot model performance...


In [ ]:
# Debugging: Test the classifier with a single sample
print("\n--- Testing classifier with a single sample ---")
if not val_df.empty:
    single_sample_text = val_df['text'].iloc[0]
    print(f"Sample text: {single_sample_text}")
    try:
        single_result = classifier(single_sample_text, candidate_labels, multi_label=False)
        print("Classifier output for single sample:")
        print(single_result)
    except Exception as e:
        print(f"Error when classifying single sample: {e}")
else:
    print("val_df is empty, cannot get a single sample.")
print("---------------------------------------------")


--- Testing classifier with a single sample ---
Sample text: तेरी गांव में दम नहीं है मोहतरमा यदि तेरी गांव में दम होता तो शरारती तत्व न लिखती। शरारती तत्व नहीं वे कटुए मुल्ले हरामी की औलाद सूअर के बच्चे थे। कोई हिन्दू यही करता तो देश की एकता को खतरा । सालों चुल्लू भर पानी में डूब मरो या सिख बन जाओ। जय माँ काली ।
Classifier output for single sample:
{'sequence': 'तेरी गांव में दम नहीं है मोहतरमा यदि तेरी गांव में दम होता तो शरारती तत्व न लिखती। शरारती तत्व नहीं वे कटुए मुल्ले हरामी की औलाद सूअर के बच्चे थे। कोई हिन्दू यही करता तो देश की एकता को खतरा । सालों चुल्लू भर पानी में डूब मरो या सिख बन जाओ। जय माँ काली ।', 'labels': ['not-offensive', 'offensive'], 'scores': [0.8854228258132935, 0.11457718163728714]}
---------------------------------------------


In [ ]:
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

# y_true is available from previous cells (cell `e72a1d2e`)
# y_pred_zero_shot is from the current zero-shot inference

print("\nClassification Report (Zero-Shot Model):")
print(classification_report(y_true, y_pred_zero_shot))

accuracy_zero_shot = accuracy_score(y_true, y_pred_zero_shot)
weighted_f1_zero_shot = f1_score(y_true, y_pred_zero_shot, average='weighted')
weighted_precision_zero_shot = precision_score(y_true, y_pred_zero_shot, average='weighted')
weighted_recall_zero_shot = recall_score(y_true, y_pred_zero_shot, average='weighted')

macro_f1_zero_shot = f1_score(y_true, y_pred_zero_shot, average='macro')
macro_precision_zero_shot = precision_score(y_true, y_pred_zero_shot, average='macro')
macro_recall_zero_shot = recall_score(y_true, y_pred_zero_shot, average='macro')

print(f"\nAccuracy (Zero-Shot): {accuracy_zero_shot:.4f}")
print(f"Weighted F1-score (Zero-Shot): {weighted_f1_zero_shot:.4f}")
print(f"Weighted Precision (Zero-Shot): {weighted_precision_zero_shot:.4f}")
print(f"Weighted Recall (Zero-Shot): {weighted_recall_zero_shot:.4f}")
print(f"Macro F1-score (Zero-Shot): {macro_f1_zero_shot:.4f}")
print(f"Macro Precision (Zero-Shot): {macro_precision_zero_shot:.4f}")
print(f"Macro Recall (Zero-Shot): {macro_recall_zero_shot:.4f}")

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate the confusion matrix for Zero-Shot
cm_zero_shot = confusion_matrix(y_true, y_pred_zero_shot)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_zero_shot, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Non-Toxic', 'Predicted Toxic'],
            yticklabels=['Actual Non-Toxic', 'Actual Toxic'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Zero-Shot Model')
plt.show()

In [ ]:
import pandas as pd

metrics_data = {
    'Metric': [
        'Accuracy',
        'Weighted F1-score',
        'Weighted Precision',
        'Weighted Recall',
        'Macro F1-score',
        'Macro Precision',
        'Macro Recall'
    ],
    'Zero-Shot Model': [
        accuracy_zero_shot,
        weighted_f1_zero_shot,
        weighted_precision_zero_shot,
        weighted_recall_zero_shot,
        macro_f1_zero_shot,
        macro_precision_zero_shot,
        macro_recall_zero_shot
    ]
}

metrics_df = pd.DataFrame(metrics_data)

print("Zero-Shot Model Performance Metrics:")
display(metrics_df)

In [ ]:
output_metrics_file = "H_Robertazero_shot_performance_metrics.xlsx"
metrics_df.to_excel(output_metrics_file, index=False)
print(f"Zero-shot roberta model performance metrics saved to {output_metrics_file}")

Zero-shot roberta model performance metrics saved to H_Robertazero_shot_performance_metrics.xlsx


### Zero-Shot Evaluation for MURIL and IndicBERT

We will now evaluate the zero-shot classification performance using `google/muril-base-cased` and `ai4bharat/indic-bert` models. It's important to note that the `zero-shot-classification` pipeline from `transformers` typically performs best with models explicitly fine-tuned on Natural Language Inference (NLI) tasks (like the `joeddav/xlm-roberta-large-xnli` model used previously). Using base models directly might yield different results.

In [ ]:
# --- Evaluate MURIL Zero-Shot Model ---
print("\n--- Evaluating google/muril-base-cased for Zero-Shot Classification ---")
muril_zero_shot_model_name = "google/muril-base-cased"
muril_zero_shot_tokenizer = AutoTokenizer.from_pretrained(muril_zero_shot_model_name)
muril_classifier = pipeline("zero-shot-classification", model=muril_zero_shot_model_name, tokenizer=muril_zero_shot_tokenizer)

y_pred_muril_zero_shot = get_zero_shot_predictions(muril_classifier, zero_shot_dataloader, candidate_labels)

accuracy_muril_zero_shot = accuracy_score(y_true, y_pred_muril_zero_shot)
weighted_f1_muril_zero_shot = f1_score(y_true, y_pred_muril_zero_shot, average='weighted')
weighted_precision_muril_zero_shot = precision_score(y_true, y_pred_muril_zero_shot, average='weighted')
weighted_recall_muril_zero_shot = recall_score(y_true, y_pred_muril_zero_shot, average='weighted')

macro_f1_muril_zero_shot = f1_score(y_true, y_pred_muril_zero_shot, average='macro')
macro_precision_muril_zero_shot = precision_score(y_true, y_pred_muril_zero_shot, average='macro')
macro_recall_muril_zero_shot = recall_score(y_true, y_pred_muril_zero_shot, average='macro')

print(f"Accuracy (MURIL Zero-Shot): {accuracy_muril_zero_shot:.4f}")
print(f"Weighted F1-score (MURIL Zero-Shot): {weighted_f1_muril_zero_shot:.4f}")

muril_metrics = [
    accuracy_muril_zero_shot,
    weighted_f1_muril_zero_shot,
    weighted_precision_muril_zero_shot,
    weighted_recall_muril_zero_shot,
    macro_f1_muril_zero_shot,
    macro_precision_muril_zero_shot,
    macro_recall_muril_zero_shot
]


--- Evaluating google/muril-base-cased for Zero-Shot Classification ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

Accuracy (MURIL Zero-Shot): 0.4234
Weighted F1-score (MURIL Zero-Shot): 0.4231


In [ ]:
# --- Evaluate IndicBERT Zero-Shot Model ---
print("\n--- Evaluating ai4bharat/indic-bert for Zero-Shot Classification ---")
indicbert_zero_shot_model_name = "ai4bharat/indic-bert"
indicbert_zero_shot_tokenizer = AutoTokenizer.from_pretrained(indicbert_zero_shot_model_name)
indicbert_classifier = pipeline("zero-shot-classification", model=indicbert_zero_shot_model_name, tokenizer=indicbert_zero_shot_tokenizer)

y_pred_indicbert_zero_shot = get_zero_shot_predictions(indicbert_classifier, zero_shot_dataloader, candidate_labels)

accuracy_indicbert_zero_shot = accuracy_score(y_true, y_pred_indicbert_zero_shot)
weighted_f1_indicbert_zero_shot = f1_score(y_true, y_pred_indicbert_zero_shot, average='weighted')
weighted_precision_indicbert_zero_shot = precision_score(y_true, y_pred_indicbert_zero_shot, average='weighted')
weighted_recall_indicbert_zero_shot = recall_score(y_true, y_pred_indicbert_zero_shot, average='weighted')

macro_f1_indicbert_zero_shot = f1_score(y_true, y_pred_indicbert_zero_shot, average='macro')
macro_precision_indicbert_zero_shot = precision_score(y_true, y_pred_indicbert_zero_shot, average='macro')
macro_recall_indicbert_zero_shot = recall_score(y_true, y_pred_indicbert_zero_shot, average='macro')

print(f"Accuracy (IndicBERT Zero-Shot): {accuracy_indicbert_zero_shot:.4f}")
print(f"Weighted F1-score (IndicBERT Zero-Shot): {weighted_f1_indicbert_zero_shot:.4f}")

indicbert_metrics = [
    accuracy_indicbert_zero_shot,
    weighted_f1_indicbert_zero_shot,
    weighted_precision_indicbert_zero_shot,
    weighted_recall_indicbert_zero_shot,
    macro_f1_indicbert_zero_shot,
    macro_precision_indicbert_zero_shot,
    macro_recall_indicbert_zero_shot
]


--- Evaluating ai4bharat/indic-bert for Zero-Shot Classification ---


Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForSequenceClassification LOAD REPORT from: ai4bharat/indic-bert
Key                              | Status     | 
---------------------------------+------------+-
sop_classifier.classifier.weight | UNEXPECTED | 
sop_classifier.classifier.bias   | UNEXPECTED | 
predictions.dense.bias           | UNEXPECTED | 
predictions.decoder.weight       | UNEXPECTED | 
predictions.bias                 | UNEXPECTED | 
predictions.dense.weight         | UNEXPECTED | 
predictions.LayerNorm.weight     | UNEXPECTED | 
predictions.decoder.bias         | UNEXPECTED | 
predictions.LayerNorm.bias       | UNEXPECTED | 
classifier.weight                | MISSING    | 
classifier.bias                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Failed to determine 'entailment' label id fr

Accuracy (IndicBERT Zero-Shot): 0.5791
Weighted F1-score (IndicBERT Zero-Shot): 0.5908


In [ ]:
# --- Combine all metrics and save to Excel ---
print("\n--- Combining and Saving All Zero-Shot Metrics ---")

# Assuming metrics_df already contains the XLM-RoBERTa and MURIL metrics
# Add IndicBERT metrics as a new column
metrics_df['IndicBERT Zero-Shot Model'] = indicbert_metrics

print("Combined Zero-Shot Model Performance Metrics (XLM-RoBERTa, MURIL, and IndicBERT):")
display(metrics_df)

final_output_metrics_file = "combined_zero_shot_performance_metrics.xlsx"
metrics_df.to_excel(final_output_metrics_file, index=False)
print(f"Combined zero-shot model performance metrics saved to {final_output_metrics_file}")

In [ ]:
# --- Re-combining all metrics to ensure MURIL is included and saving to Excel ---
print("\n--- Re-combining All Zero-Shot Metrics including MURIL and IndicBERT ---")

# Re-create metrics_df from scratch to ensure all models are included correctly
metrics_data_all = {
    'Metric': [
        'Accuracy',
        'Weighted F1-score',
        'Weighted Precision',
        'Weighted Recall',
        'Macro F1-score',
        'Macro Precision',
        'Macro Recall'
    ],
    'Zero-Shot Model': [
        accuracy_zero_shot,
        weighted_f1_zero_shot,
        weighted_precision_zero_shot,
        weighted_recall_zero_shot,
        macro_f1_zero_shot,
        macro_precision_zero_shot,
        macro_recall_zero_shot
    ]
}
metrics_df_final = pd.DataFrame(metrics_data_all)

# Add MURIL metrics
metrics_df_final['MURIL Zero-Shot Model'] = muril_metrics

# Add IndicBERT metrics
metrics_df_final['IndicBERT Zero-Shot Model'] = indicbert_metrics

print("Final Combined Zero-Shot Model Performance Metrics (XLM-RoBERTa, MURIL, and IndicBERT):")
display(metrics_df_final)

final_output_metrics_file = "combined_zero_shot_performance_metrics.xlsx"
metrics_df_final.to_excel(final_output_metrics_file, index=False)
print(f"Final combined zero-shot model performance metrics saved to {final_output_metrics_file}")